# v76 — refiner checkpoint ensemble 評価 (fold2 OOF, Colab)

独立に学習・検証された refiner ckpt の確率マップ平均を、
reviewed pipeline (prod refiner・fold2 OOF 235ファイル・frozen operating
point conf0.26/minarea20/drop0.90) で検証する推論のみの実験。学習なし。

| arm | prob map | members |
|---|---|---|
| `baseline` | prod single-view | fold2_resume (b1c0f353) |
| `ens3` | 3 member 平均 | baseline + v74 scratch_s8407 + v74 ssl_s8407 |
| `ens3f` | 3 member × flip4 TTA 平均 | 同上 (ensemble + TTA 積み上げ) |

**事前固定ゲート** (RESEARCH_LOG §2.31):
- delta(ens3f − baseline)@PQ0.45 ≥ +0.005 → 昇格候補 (別fold/LB 確認へ)
- [0,+0.005) → ノイズ棄却 / <0 → 棄却
- secondary: ens3 − baseline で ensemble 単独効果を分離
- sanity: baseline ≈ v74/v75 実績 0.444238 ±0.001

**Provenance pin:** `V73_PROVENANCE_SHA256` (v73-v75 と同一 manifest)。
member は manifest 宣言 (baseline) または ckpt 内 metadata (v74 系) で
検証 — unknown bytes は resolve_binding_role が拒否する。
Drive `solar_filament_v35/kaggle.json` で kaggle 認証 (shingo257)。
eval 終了時に Drive `solar_filament_v76/work/latest/` へ sha256 検証付きミラー。


In [ ]:
import os, re, json, subprocess, sys, hashlib, shutil, time
from pathlib import Path
# Reviewed provenance pin is supplied externally — env var, Colab secret,
# Kaggle secret — or declared interactively. Never derived from artifacts.
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256 (reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_PROVENANCE_SHA256 after independent provenance review')

# [0] Drive mount — primary data source (same pattern as all past colab runs)
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
V74 = Path('/content/drive/MyDrive/solar_filament_v74/work/latest')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'test_dir': COMP / 'test/test_images',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold2_resume.pt',
    'ens_scratch_ckpt': V74 / 'edgeatt_scratch_s8407_fold2.pt',
    'ens_ssl_ckpt': V74 / 'edgeatt_ssl_s8407_fold2.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_2/weights/best.pt',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
train_json, test_dir, preproc_dir = (inputs['train_json'], inputs['test_dir'],
                                     inputs['preproc_dir'])
folds_csv, baseline_ckpt, yolo_ckpt = (inputs['folds_csv'], inputs['baseline_ckpt'],
                                      inputs['yolo_ckpt'])
ens_members = [inputs['baseline_ckpt'], inputs['ens_scratch_ckpt'],
             inputs['ens_ssl_ckpt']]
if not any(test_dir.iterdir()):
    raise RuntimeError('test_images dir is empty')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual} (Drive artifacts/grouped_folds.csv)')
print('folds_csv sha256 OK (true experimental split)')

# [1] Kaggle auth + code/manifest download
os.makedirs('/root/.kaggle', exist_ok=True)
cred = inputs['kaggle.json'].read_bytes()
print('auth: Drive kaggle.json')
try:
    _u = json.loads(cred)['username']
except Exception:
    _u = '?'
print('kaggle user:', _u)
Path('/root/.kaggle/kaggle.json').write_bytes(cred)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics', 'pycocotools'], check=True)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _dl(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('download failed rc=%s: %s\n%s\n%s' % (
            r.returncode, cmd, r.stdout[-800:], r.stderr[-800:]))
def _sha(p):
    d = hashlib.sha256()
    with open(p, 'rb') as h:
        for b in iter(lambda: h.read(1 << 20), b''):
            d.update(b)
    return d.hexdigest()

_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code',
     '-p', str(IN), '--unzip', '-q'])
_dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-v73-assets',
     '-f', 'v73_reviewed_provenance.json', '-p', str(IN)])

code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
from v73_split_provenance import (load_reviewed_manifest, load_fold_mapping,
                                  validate_checkpoint_binding)
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 2,
                  manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)

# [2] TTA eval variant — embedded copy of repo scratch/run_fold_oof_eval_tta.py
TTA_SCRIPT = '"""fold N OOF eval with refiner flip-4 TTA + topology guard (v75).\n\nrun_fold_oof_eval.py と同一の proposals/postprocess/PQ 計算を共有し、\nrefiner 確率マップの計算だけを variant にする paired comparison。\narm 指定は `name=ckpt_path[:mode]` (mode: none|flip4|flip4_guard,\ndefault none)。\n\nflip4: 320px crop を identity/hflip/vflip/hvflip の4view で推論し、\n逆変換して平均した確率マップを使う (Fususu handoff 由来)。\n\nflip4_guard: flip4 平均マップが single-view の topology を保存する時だけ\n採用し、違反時は single-view にフォールバック。guard は crop 解像度の\n0.5 二値マップで判定 (事前固定, RESEARCH_LOG §2.30):\n  - 面積比 tta/single が [0.85, 1.15] 内\n  - 連結成分数 (>=16px) の差が <=1\n  - IoU(single_bin, tta_bin) >= 0.85\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\ntry:\n    from scratch.run_fold_oof_eval import (  # noqa: E402\n        CROP_SIZE, CONF, MIN_AREA, DROP_THRESHOLD, EVAL_SIZE,\n        load_models, load_ownership, to_rle, iou_matrix, mask_square_box,\n        result_masks)\n    from scratch.v73_split_provenance import (  # noqa: E402\n        load_fold_mapping, load_reviewed_manifest, sha256_file,\n        validate_checkpoint_binding, validate_training_corpus_provenance)\nexcept ImportError:\n    from run_fold_oof_eval import (  # noqa: E402\n        CROP_SIZE, CONF, MIN_AREA, DROP_THRESHOLD, EVAL_SIZE,\n        load_models, load_ownership, to_rle, iou_matrix, mask_square_box,\n        result_masks)\n    from v73_split_provenance import (  # noqa: E402\n        load_fold_mapping, load_reviewed_manifest, sha256_file,\n        validate_checkpoint_binding, validate_training_corpus_provenance)\n\ntry:\n    from src.metrics_official import PanopticQuality  # noqa: E402\nexcept ImportError:\n    from metrics_official import PanopticQuality  # noqa: E402\n\nTTA_MODES = ("none", "flip4", "flip4_guard")\nGUARD_AREA_RATIO = (0.85, 1.15)\nGUARD_MAX_CC_DELTA = 1\nGUARD_MIN_IOU = 0.85\nGUARD_MIN_CC_AREA = 16\n\n\n@torch.no_grad()\ndef _crop_prob(refiner, crop):\n    tensor = (torch.from_numpy(np.ascontiguousarray(crop.transpose(2, 0, 1))[None])\n              .float().cuda() / 255.0)\n    return refiner(tensor).sigmoid()[0, 0].cpu().numpy()\n\n\ndef _flip(arr, hf, vf):\n    out = arr[:, ::-1] if hf else arr\n    return out[::-1] if vf else out\n\n\ndef _cc_count(binary):\n    count, _, stats, _ = cv2.connectedComponentsWithStats(\n        binary.astype(np.uint8), connectivity=8)\n    return sum(1 for i in range(1, count)\n               if stats[i, cv2.CC_STAT_AREA] >= GUARD_MIN_CC_AREA)\n\n\ndef topology_ok(single_prob, tta_prob):\n    """事前固定 guard: tta 二値マップが single-view topology を保存するか。"""\n    single = single_prob >= 0.5\n    tta = tta_prob >= 0.5\n    s_area, t_area = int(single.sum()), int(tta.sum())\n    if s_area == 0 or t_area == 0:\n        return s_area == t_area\n    if not GUARD_AREA_RATIO[0] <= t_area / s_area <= GUARD_AREA_RATIO[1]:\n        return False\n    if abs(_cc_count(single) - _cc_count(tta)) > GUARD_MAX_CC_DELTA:\n        return False\n    union = int((single | tta).sum())\n    return int((single & tta).sum()) / union >= GUARD_MIN_IOU\n\n\n@torch.no_grad()\ndef refine_probability_tta(refiner, image, proposal, mode):\n    """(prob_map, box, fell_back) を返す。\n\n    mode=none は run_fold_oof_eval.refine_probability と同一経路。\n    fell_back は flip4_guard で guard 違反により single-view を採用した時のみ\n    True。\n    """\n    x0, y0, x1, y1 = mask_square_box(proposal)\n    crop = cv2.resize(image[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE),\n                      interpolation=cv2.INTER_LINEAR)\n    single = _crop_prob(refiner, crop)\n    fell_back = False\n    if mode == "none":\n        use = single\n    else:\n        acc = single.astype(np.float64)\n        for hf, vf in ((1, 0), (0, 1), (1, 1)):\n            acc += _flip(_crop_prob(refiner, _flip(crop, hf, vf)), hf, vf)\n        tta = acc / 4.0\n        fell_back = mode == "flip4_guard" and not topology_ok(single, tta)\n        use = single if fell_back else tta\n    return (cv2.resize(use.astype(np.float32), (x1 - x0, y1 - y0),\n                       interpolation=cv2.INTER_LINEAR), (x0, y0, x1, y1),\n            fell_back)\n\n\ndef arm_probability(models, image, proposal, mode):\n    """ensemble arm の確率マップ: 各 member の prob map (mode 適用済) を\n    平均する。box は proposal 由来で全 member 共通。fell_back は\n    member 全体が single-view にフォールバックした時のみ True。"""\n    probs, box, fbs = [], None, []\n    for model in models:\n        prob, box, fb = refine_probability_tta(model, image, proposal, mode)\n        probs.append(prob.astype(np.float64))\n        fbs.append(fb)\n    return (np.mean(probs, axis=0).astype(np.float32), box,\n            all(fbs) if fbs else False)\n\n\ndef resolve_binding_role(path, name, manifest, checkpoint):\n    """Arm 名が manifest の role ならそのまま使う。無ければ、ファイルが\n    reviewed checkpoint のどれかとバイト一致する時だけその role を継承する\n    (tta4/tta4g・ensemble member が baseline ckpt を共有するケース)。\n    ckpt 自体が fold/folds_sha256 metadata を持つ場合は arm 名のまま\n    validate_checkpoint_binding の metadata 検証に委ねる\n    (v73/v74 で学習された ckpt)。どれにも該当しない未知バイトは拒否する。"""\n    declared = manifest["checkpoints"]\n    if name in declared:\n        return name\n    digest = sha256_file(path)\n    matched = sorted(role for role, expected in declared.items()\n                     if expected == digest)\n    if matched:\n        return matched[0]\n    if (isinstance(checkpoint, dict)\n            and ("fold" in checkpoint or "folds_sha256" in checkpoint)):\n        return name\n    raise ValueError(f"checkpoint provenance missing: {name} "\n                     "(bytes match no reviewed checkpoint binding)")\n\n\ndef parse_arm(spec):\n    """name=ckpt[:mode] または name=ckpt1+ckpt2+...[:mode] (ensemble arm)。"""\n    name, sep, rest = spec.partition("=")\n    path, _, mode = rest.rpartition(":")\n    if mode not in TTA_MODES:\n        path, mode = rest, "none"\n    paths = [Path(p) for p in path.split("+") if p]\n    if (not name or name == "yolo" or not paths\n            or not all(p.exists() for p in paths)):\n        raise ValueError(f"--refiner name=path[:mode] が不正: {spec}")\n    return name, paths, mode\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train-json", type=Path, required=True)\n    parser.add_argument("--preproc-dir", type=Path, required=True)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--folds-csv", type=Path, required=True)\n    parser.add_argument("--provenance-manifest", type=Path, required=True)\n    parser.add_argument("--provenance-sha256", required=True)\n    parser.add_argument("--fold", type=int, required=True, choices=(0, 1, 2))\n    parser.add_argument("--yolo", type=Path, required=True)\n    parser.add_argument("--refiner", action="append", required=True,\n                        help="name=ckpt_path[:none|flip4|flip4_guard]")\n    parser.add_argument("--model", type=str, default="edgeatt",\n                        choices=("unet", "edgeatt"))\n    parser.add_argument("--thresholds", type=str, default="0.35,0.40,0.45,0.50")\n    parser.add_argument("--conf", type=float, default=CONF)\n    parser.add_argument("--limit", type=int, default=0)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    manifest = load_reviewed_manifest(\n        args.provenance_manifest, args.provenance_sha256, args.fold)\n    if args.limit < 0:\n        raise ValueError("--limit must be non-negative")\n    thresholds = [float(t) for t in args.thresholds.split(",")]\n    seen = set()\n    arms = []\n    for spec in args.refiner:\n        name, path, mode = parse_arm(spec)\n        if name in seen:\n            raise ValueError(f"duplicate arm: {name}")\n        seen.add(name)\n        arms.append((name, path, mode))\n\n    folds_sha256 = hashlib.sha256(args.folds_csv.read_bytes()).hexdigest()\n    with open(args.train_json, encoding="utf-8") as handle:\n        coco = json.load(handle)\n    images = {i["id"]: i for i in coco["images"]}\n    annotations = defaultdict(list)\n    for ann in coco["annotations"]:\n        annotations[ann["image_id"]].append(ann)\n    file_to_ids = defaultdict(list)\n    for image_id, info in images.items():\n        file_to_ids[info["file_name"]].append(image_id)\n    fold_of_file = load_fold_mapping(\n        args.folds_csv, file_to_ids, args.fold, manifest["folds_sha256"])\n    yolo_sha256 = validate_checkpoint_binding(args.yolo, "yolo", manifest)\n    checked_arms = {}\n    for name, paths, mode in arms:\n        members = []\n        for path in paths:\n            checkpoint = torch.load(path, map_location="cpu",\n                                    weights_only=True)\n            role = resolve_binding_role(path, name, manifest, checkpoint)\n            digest = validate_checkpoint_binding(\n                path, role, manifest, checkpoint)\n            provenance = checkpoint.get(\n                "init_report", {}).get("corpus_provenance")\n            if (str(checkpoint.get("init", "")).startswith("ssl:")\n                    or provenance is not None):\n                validate_training_corpus_provenance(\n                    provenance, coco, args.test_dir)\n            members.append((checkpoint, digest))\n        checked_arms[name] = (members, mode)\n    oof_files = [n for n in sorted(file_to_ids)\n                 if fold_of_file.get(n) == args.fold]\n    if args.limit:\n        oof_files = oof_files[:args.limit]\n    print({"fold": args.fold, "oof_files": len(oof_files),\n           "arms": [(n, m) for n, _, m in arms],\n           "folds_sha256": folds_sha256[:16]}, flush=True)\n\n    MediSegUNet, MediSegEdgeAttUNet = load_models()\n    apply_ownership = load_ownership()\n    model_cls = MediSegEdgeAttUNet if args.model == "edgeatt" else MediSegUNet\n    from pycocotools import mask as mask_utils\n    from ultralytics import YOLO\n\n    gt_rles = {image_id: [to_rle(a, int(info["height"]), int(info["width"]),\n                                 mask_utils)\n                          for a in annotations[image_id]\n                          if not a.get("iscrowd", 0)]\n               for image_id, info in images.items()}\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    refiners = {}\n    for name, _, mode in arms:\n        members, _ = checked_arms.pop(name)\n        models = []\n        for ckpt, _digest in members:\n            model = model_cls(3, 1, False).cuda().eval()\n            model.load_state_dict(ckpt.get("best_model") or ckpt["model"],\n                                  strict=True)\n            models.append(model)\n        refiners[name] = {"models": models,\n                          "sha256": [d for _, d in members],\n                          "tta_mode": mode,\n                          "best_dice": [c.get("best_dice")\n                                        for c, _ in members],\n                          "init": [c.get("init", "pretrained")\n                                   for c, _ in members]}\n        print(f"arm {name}: members={len(models)} tta={mode} "\n              f"best_dice={refiners[name][\'best_dice\']}", flush=True)\n\n    yolo_model = YOLO(str(args.yolo))\n    started = time.time()\n\n    proposals_by_file = {}\n    for name in oof_files:\n        image_path = args.preproc_dir / name\n        result = yolo_model.predict(\n            str(image_path), imgsz=1280, conf=0.08, iou=0.55,\n            agnostic_nms=True, retina_masks=True, verbose=False)[0]\n        masks, confidences = result_masks(result)\n        keep = [i for i, (m, s) in enumerate(zip(masks, confidences))\n                if s >= args.conf and m.sum() >= MIN_AREA]\n        proposals_by_file[name] = [masks[i] for i in keep]\n    print(f"proposals done: {time.time() - started:.0f}s", flush=True)\n\n    report = {"fold": args.fold, "folds_sha256": folds_sha256,\n              "provenance_manifest_sha256": args.provenance_sha256,\n              "yolo_sha256": yolo_sha256,\n              "conf": args.conf, "min_area": MIN_AREA,\n              "drop_threshold": DROP_THRESHOLD, "thresholds": thresholds,\n              "tta_guard": {"area_ratio": list(GUARD_AREA_RATIO),\n                            "max_cc_delta": GUARD_MAX_CC_DELTA,\n                            "min_iou": GUARD_MIN_IOU,\n                            "min_cc_area": GUARD_MIN_CC_AREA},\n              "oof_files": len(oof_files), "arms": {}}\n    for arm_name, arm in refiners.items():\n        meters = {t: PanopticQuality() for t in thresholds}\n        fallback = {"applied": 0, "fallback": 0}\n        for name in oof_files:\n            image_path = args.preproc_dir / name\n            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n            mode = arm["tta_mode"]\n            results = [arm_probability(arm["models"], image, m, mode)\n                       for m in proposals_by_file[name]]\n            prob_maps = [(prob, box) for prob, box, _ in results]\n            fallback["applied"] += sum(1 for *_, fb in results if not fb)\n            fallback["fallback"] += sum(1 for *_, fb in results if fb)\n            for threshold in thresholds:\n                refined = []\n                for (probability, box), proposal in zip(\n                        prob_maps, proposals_by_file[name]):\n                    x0, y0, x1, y1 = box\n                    mask = np.zeros_like(proposal, dtype=bool)\n                    mask[y0:y1, x0:x1] = probability >= threshold\n                    refined.append(mask & proposal)\n                predictions = apply_ownership(\n                    refined, np.ones(len(refined)),\n                    rule="drop_contained_then_area_asc",\n                    containment_threshold=DROP_THRESHOLD,\n                    min_area=MIN_AREA, canvas=EVAL_SIZE)\n                pred_rles = (mask_utils.encode(np.asfortranarray(\n                    np.stack(predictions, axis=-1).astype(np.uint8)))\n                    if predictions else [])\n                for image_id in file_to_ids[name]:\n                    truth = gt_rles[image_id]\n                    matrix = iou_matrix(truth, pred_rles, mask_utils)\n                    meters[threshold].add_matrix(matrix, len(truth),\n                                                 len(pred_rles))\n        report["arms"][arm_name] = {\n            "ckpt_sha256": (arm["sha256"][0]\n                            if len(arm["sha256"]) == 1 else arm["sha256"]),\n            "member_sha256": arm["sha256"], "init": arm["init"],\n            "tta_mode": arm["tta_mode"],\n            "tta_fallback": fallback if arm["tta_mode"] == "flip4_guard" else None,\n            "best_crop_dice": arm["best_dice"],\n            "by_threshold": {str(t): meters[t].summary() for t in thresholds},\n        }\n        report["arms"][arm_name]["pq_by_threshold"] = {\n            str(t): round(meters[t].score, 6) for t in thresholds}\n        print({arm_name: report["arms"][arm_name]["pq_by_threshold"]},\n              flush=True)\n\n    report["minutes"] = round((time.time() - started) / 60, 1)\n    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2),\n                        encoding="utf-8")\n    print(json.dumps({a: r["pq_by_threshold"]\n                      for a, r in report["arms"].items()},\n                     ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path(code_dir / 'run_fold_oof_eval_tta.py').write_text(TTA_SCRIPT)
import ast; ast.parse(TTA_SCRIPT)
print('tta eval script written:', code_dir / 'run_fold_oof_eval_tta.py')
for name, p in [('preproc', preproc_dir), ('folds', folds_csv),
                ('baseline', baseline_ckpt), ('yolo', yolo_ckpt),
                ('ens_members', ens_members),
                ('train_json', train_json), ('code', code_dir)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

MIRROR_DIR = Path('/content/drive/MyDrive/solar_filament_v76/work/latest')
def mirror():
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Drive not mounted')
    MIRROR_DIR.mkdir(parents=True, exist_ok=True)
    digests = {}
    for p in sorted(WORK.rglob('*')):
        if p.is_file() and 'input' not in p.relative_to(WORK).parts:
            dst = MIRROR_DIR / p.relative_to(WORK)
            expected = _sha(p)
            if dst.exists() and _sha(dst) == expected:
                digests[str(p.relative_to(WORK))] = expected; continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            if _sha(dst) != expected:
                raise RuntimeError(f'Drive mirror sha mismatch: {p.name}')
            digests[str(p.relative_to(WORK))] = expected
            print('mirrored', p.name, p.stat().st_size)
    (MIRROR_DIR / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
    print('mirror DONE ->', MIRROR_DIR)


In [ ]:
%%time
# [eval] 3 arm — 同一 YOLO 提案で refiner member 構成だけを変える
ens_spec = '+'.join(str(p) for p in ens_members)
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval_tta.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--yolo', str(yolo_ckpt), '--model', 'edgeatt',
                    '--refiner', f'baseline={baseline_ckpt}:none',
                    '--refiner', f'ens3={ens_spec}:none',
                    '--refiner', f'ens3f={ens_spec}:flip4',
                    '--out', str(WORK / 'v76_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v76_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')
mirror()


In [ ]:
# 結果サマリ — 事前固定ゲート判定 (§2.31)
rep = json.load(open(WORK / 'v76_eval.json'))
arms = rep['arms']
pq = lambda a: arms[a]['pq_by_threshold']['0.45']
print(json.dumps({a: r['pq_by_threshold'] for a, r in arms.items()}, indent=2))
d_f = round(pq('ens3f') - pq('baseline'), 6)
d_e = round(pq('ens3') - pq('baseline'), 6)
gate = {
  'baseline_pq_0.45': pq('baseline'),
  'sanity_baseline_near_v75': abs(pq('baseline') - 0.444238) < 0.001,
  'baseline_diff_vs_v75': round(pq('baseline') - 0.444238, 6),
  'ens3_pq': pq('ens3'), 'ens3f_pq': pq('ens3f'),
  'delta_ens3f_minus_baseline': d_f,
  'delta_ens3_minus_baseline': d_e,
  'decision': ('promote_candidate' if d_f >= 0.005
               else 'reject' if d_f < 0 else 'reject_noise'),
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v76_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))
mirror()
